In [1]:
# 1. Connexion au Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. On s'assure que les bibliothèques nécessaires sont là
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm # Pour avoir une barre de progression

Mounted at /content/drive


In [2]:
# 1. Processeur
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Chargement DINOv2 Large
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device)

# Activation du mode "Vitesse maximale" si GPU présent
if device.type == 'cuda':
    model = model.half()
    print("Mode demi-précision (FP16) activé.")

model.eval()

import numpy as np

# 3. Transformation optimisée pour DINOv2 (Multiple de 14 et arrondis au plus proche)
def get_dinov2_size(width, height=518):
    # On force la hauteur à 518 (déjà multiple de 14)
    # On arrondit la largeur au multiple de 14 le plus proche
    new_w = int(round(width / 14) * 14)
    return [height, new_w]

transform = T.Compose([
    # Lambda qui ajuste la taille dynamiquement par image
    T.Lambda(lambda img: T.functional.resize(img, get_dinov2_size(img.size[0], img.size[1]))),
    T.ToTensor(),
    # Normalisation standard pour les modèles Vision Transformer (DINO/ViT)
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # A CONSERVER !! + 14 % d'accuracy
    ])

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:05<00:00, 206MB/s]


Mode demi-précision (FP16) activé.


In [3]:
# --- CONFIGURATION DES CHEMINS ---
# path vers les spectros zippés
path_zip = "/content/drive/MyDrive/audio_classification_sam/data/data_for_prediction/3s_pred3.zip"

# C'est ici qu'on définit l'endroit où on va extraire (le SSD local de Colab)
path_imgs = "/content/spectros_local/3s_pred3/3s_pred3"

# path vers le csv résulat
path_csv_final = "/content/drive/MyDrive/audio_classification_sam/embeddings/3s_pred3.csv"

# --- EXÉCUTION ---
# 1. Création du dossier local
!mkdir -p {path_imgs}

# 2. On lance l'unzip
print(f"Début de l'extraction de : {path_zip}")

!unzip -q "{path_zip}" -d {path_imgs}

print("Extraction terminée")

Début de l'extraction de : /content/drive/MyDrive/audio_classification_sam/data/data_for_prediction/3s_pred3.zip
Extraction terminée


In [4]:
# --- CELLULE DE NETTOYAGE DES DIMENSIONS --- (sinon bug de dinoV2 si une image a une largeur erronnée)
import os
from PIL import Image
from collections import Counter

# 1. On s'assure d'utiliser le bon chemin
target_dir = path_imgs
print(f"Analyse des fichiers dans : {target_dir}")

# Gestion automatique du sous-dossier apres unzip (s'il y a un dossier racine supplementaire)
if os.path.exists(target_dir):
    content = [f for f in os.listdir(target_dir) if not f.startswith('.')]
    if len(content) == 1 and os.path.isdir(os.path.join(target_dir, content[0])):
        target_dir = os.path.join(target_dir, content[0])
        print(f"Utilisation du sous-dossier detecte : {target_dir}")


# 2. Scan rapide des dimensions
all_files = [f for f in os.listdir(target_dir) if f.endswith('.png')]
widths = []


print("Scan des dimensions en cours...")
for f in tqdm(all_files):
    with Image.open(os.path.join(target_dir, f)) as img:
        widths.append(img.size[0])


# 3. Identification de la largeur majoritaire
data_counts = Counter(widths)
# Ajout d'une verification au cas ou aucune image n'est traitee apres le scan
if not data_counts:
    raise ValueError("Aucune image valide trouvee pour l'analyse des dimensions. Verifiez le chemin ou les fichiers.")
main_width = data_counts.most_common(1)[0][0]
print(f"\nLargeur standard identifiee : {main_width}px")


# 4. Suppression des images dont la largeur est incorrecte
deleted_count = 0
for f in all_files:
    path = os.path.join(target_dir, f)
    with Image.open(path) as img:
        if img.size[0] != main_width:
            img.close() # On ferme le flux avant de supprimer
            os.remove(path)
            print(f"Fichier supprime (largeur incorrecte {img.size[0]}px) : {f}")
            deleted_count += 1


if deleted_count == 0:
    print("Aucune anomalie detectee. Toutes les images sont uniformes.")
else:
    print(f"\nNettoyage termine : {deleted_count} fichier(s) supprime(s).")

    # Mise a jour du dataset pour la cellule suivante
    # Le dataset SpectroDataset n'est pas defini dans cette cellule, donc ce bloc pourrait causer une erreur ici.
    # Si 'dataset' est utilise dans une cellule ulterieure, il sera reinitialise.
    # Ce commentaire sera conserve pour l'explication mais retire du code si la classe est definie ailleurs.
    # dataset = SpectroDataset(target_dir, transform=transform)
    # print(f"Nouveau nombre d'images pour l'extraction : {len(dataset)}")


Analyse des fichiers dans : /content/spectros_local/3s_pred3/3s_pred3
Utilisation du sous-dossier detecte : /content/spectros_local/3s_pred3/3s_pred3/3s_prediction_mel_spectrograms
Scan des dimensions en cours...


100%|██████████| 67033/67033 [01:10<00:00, 950.45it/s]



Largeur standard identifiee : 522px
Aucune anomalie detectee. Toutes les images sont uniformes.


In [5]:
import csv
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION DU CALCUL ---
BATCH_SIZE = 128
NUM_WORKERS = 2

# --- DÉTECTION AUTOMATIQUE PROFONDE ---
import glob
# On cherche n'importe quel fichier PNG dans l'arborescence complète
found_files = glob.glob(os.path.join(path_imgs, "**/*.png"), recursive=True)

if not found_files:
    raise FileNotFoundError(f" Aucune image trouvée dans {path_imgs}. Vérifie l'extraction ZIP.")

# On définit le dossier cible comme étant celui qui contient les images
target_dir = os.path.dirname(found_files[0])
print(f" Images localisées dans : {target_dir}")
print(f" Total d'images trouvées : {len(found_files)}")

checkpoint_csv = path_csv_final

# Gestion automatique du sous-dossier apres unzip
if os.path.exists(target_dir):
    content = [f for f in os.listdir(target_dir) if not f.startswith('.')]
    if len(content) == 1 and os.path.isdir(os.path.join(target_dir, content[0])):
        target_dir = os.path.join(target_dir, content[0])
        print(f"Utilisation du sous-dossier : {target_dir}")

# 1. Preparation de la reprise
processed_files = set()
if os.path.exists(checkpoint_csv):
    try:
        existing_df = pd.read_csv(checkpoint_csv, usecols=["filename"])
        processed_files = set(existing_df["filename"].tolist())
        print(f"Reprise detectee : {len(processed_files)} images deja traitees.")
    except Exception as e:
        print(f"Creation d'un nouveau fichier (Fichier actuel illisible ou absent)")

# 2. Filtrage du Dataset
class FilteredSpectroDataset(Dataset):
    def __init__(self, img_dir, processed_set, transform=None):
        self.img_dir = img_dir
        if not os.path.exists(img_dir):
            raise FileNotFoundError(f"Dossier introuvable : {img_dir}")
        all_f = sorted([f for f in os.listdir(img_dir) if f.endswith('.png')])
        self.img_names = [f for f in all_f if f not in processed_set]
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.img_names[idx]

# Initialisation
dataset_todo = FilteredSpectroDataset(target_dir, processed_files, transform=transform)
dataloader = DataLoader(dataset_todo, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Reste a traiter : {len(dataset_todo)} images.")

# 3. Boucle d'extraction
model.to(device)
model.eval()

# Vérification si le fichier existe et n'est pas vide pour décider de l'écriture du header
file_exists = os.path.isfile(checkpoint_csv) and os.path.getsize(checkpoint_csv) > 0

with open(checkpoint_csv, mode='a', newline='') as f:
    writer = csv.writer(f)
    header_done = file_exists

    with torch.no_grad():
        for i, (batch_imgs, batch_names) in enumerate(tqdm(dataloader, desc="Extraction")):
            batch_imgs = batch_imgs.to(device)
            if device.type == 'cuda':
                batch_imgs = batch_imgs.half()

            # Inference
            outputs = model.forward_features(batch_imgs)
            embeddings = outputs["x_norm_clstoken"].cpu().float().numpy()

            # Ecriture
            for j in range(len(batch_names)):
                if not header_done:
                    # Reconstruction dynamique du header avec les dimensions réelles
                    header = ["filename"] + [f"dim_{k}" for k in range(embeddings.shape[1])]
                    writer.writerow(header)
                    header_done = True

                row = [batch_names[j]] + embeddings[j].tolist()
                writer.writerow(row)

            # Securite : Sauvegarde physique tous les 10 batches
            if (i + 1) % 10 == 0:
                f.flush()
                os.fsync(f.fileno())

print(f"Extraction terminee. Fichier enregistre : {checkpoint_csv}")

# --- SÉCURITÉ TRANSFERT GOOGLE DRIVE ---

try:
    print("Synchronisation des données vers Google Drive...")
    # On force l'écriture des buffers et la synchronisation physique
    from google.colab import drive
    drive.flush_and_unmount()
    print("Synchronisation réussie. Drive démonté.")
except Exception as e:
    print(f"Erreur lors de la synchronisation : {e}")

# --- DÉCONNEXION AUTOMATIQUE ---

import time
# On laisse une petite marge de 5 secondes par précaution
time.sleep(5)

print("Fermeture de la session Colab...")
from google.colab import runtime
runtime.unassign()


 Images localisées dans : /content/spectros_local/3s_pred3/3s_pred3/3s_prediction_mel_spectrograms
 Total d'images trouvées : 67033
Reste a traiter : 67033 images.


Extraction: 100%|██████████| 524/524 [1:16:23<00:00,  8.75s/it]


Extraction terminee. Fichier enregistre : /content/drive/MyDrive/audio_classification_sam/embeddings/3s_pred3.csv
Synchronisation des données vers Google Drive...
Synchronisation réussie. Drive démonté.
Fermeture de la session Colab...
